# 03b — Reuters Leakage Validation (Ablation Experiment)

**Project:** Fake News Detection using Deep Learning (BCA Major Project)
**Phase:** Additional validation experiment, attached to Phase 4 — does **not** replace or
modify the official baseline.

## Why this experiment exists

The official baseline (`notebooks/03_baseline_model.ipynb`) reached ~98.24% accuracy. Its
Feature Importance section found something that needed investigating rather than celebrating:
**"reuters" is the single strongest word predicting "Real"**, even though Stage 2 of
preprocessing already stripped the leading `"(Reuters) - "` dateline
(`docs/label_leakage_analysis.md`). Investigation traced this to **5,186 of 38,638 rows (13.4%)**
still containing the word "reuters" elsewhere in the article body — secondary datelines,
correction notices, or in-body citations the prefix-stripping regex doesn't reach.

This notebook runs a controlled experiment to answer one specific question: **is that residual
"reuters" mention actually propping up the baseline's accuracy (i.e. is it still a shortcut/label
leakage), or is the model's strong performance built on other, more genuine signal?**

## What this notebook does *not* do

- It does **not** retrain, modify, or overwrite `models/baseline/` — the official baseline
  artifacts are only ever *loaded* (read-only) for comparison, never refit.
- It does **not** touch `dataset/processed/03_preprocessed.csv` — a new, in-memory-only column
  (`baseline_text_no_reuters`) is derived from it for this experiment alone.
- It does **not** change the train/validation/test split, the random seed, the TF-IDF
  parameters, or the Logistic Regression parameters — every one of those is reused unchanged
  from `training/split.py` and `training/baseline.py`. **The only variable changed is whether
  the word "reuters" is present in the text.** That single-variable control is what makes this
  a valid ablation experiment rather than just "a different model."


In [1]:
import sys
import time
from pathlib import Path

import joblib
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append(str(Path.cwd().parent))

from config.settings import (
    BASELINE_MODEL_DIR,
    EXPERIMENTS_LOG_FILE,
    REPORT_FIGURES_DIR,
    REPORT_TABLES_DIR,
    STAGE3_PREPROCESSED_FILE,
    TOP_N_FEATURES,
)
from training.baseline import build_tfidf_vectorizer, train_logistic_regression
from training.split import stratified_three_way_split
from preprocessing.text_cleaning import remove_word_token, normalize_whitespace
from evaluation.metrics import (
    compute_classification_metrics,
    plot_confusion_matrix,
    plot_roc_curve,
    save_classification_report,
)
from evaluation.significance import mcnemar_test
from evaluation.feature_importance import get_top_features, plot_top_features
from evaluation.experiment_log import log_experiment

pd.set_option("display.max_colwidth", 100)
REPORT_FIGURES_DIR.mkdir(parents=True, exist_ok=True)
REPORT_TABLES_DIR.mkdir(parents=True, exist_ok=True)


## Educational background

### What is an ablation study?

An **ablation study** removes (or "ablates") one specific component, feature, or piece of
information from a system, retrains under otherwise identical conditions, and measures the
effect. The name comes from surgery ("to ablate" = to remove tissue) — the method carries over
directly: if removing a part changes the outcome a lot, that part mattered; if removing it
changes almost nothing, it wasn't doing the work you thought it was.

### Why is this experiment scientifically useful?

Feature importance (in the official baseline notebook) showed *correlation* — "reuters" has a
large coefficient. It did not, by itself, prove *how much that one word actually contributes to
the model's overall accuracy*. A large coefficient on one word is consistent with two very
different stories:

1. The model is heavily *dependent* on that one word (remove it → accuracy drops noticeably),
   which would mean the ~98% result was partly an illusion built on a shortcut.
2. The model has *many* correlated signals and this word is just the strongest *individual*
   one, but far from the *only* one (remove it → the model reallocates weight to other,
   already-present signals, and accuracy barely moves).

Only an ablation experiment — actually removing the word and re-measuring — can distinguish
between these two stories. This is the standard scientific method applied to a model: form a
hypothesis ("reuters mentions inflate our accuracy"), design a controlled test that changes
exactly one variable, and let the measured result confirm or refute the hypothesis.

### Why is verifying potential leakage better than simply reporting the highest possible accuracy?

Because a headline accuracy number, on its own, cannot be trusted at face value — this exact
project has already demonstrated that twice (the `subject` column reached 100% accuracy alone;
the Reuters dateline reached 99.6% alone; see `docs/label_leakage_analysis.md`). A student
project that reports "98% accuracy" without investigating *why* invites a very reasonable viva
question: "are you sure that's not another shortcut?" Running this ablation, and reporting its
result honestly (whichever way it comes out), is what turns a suspiciously good number into a
*defensible* one — the report can say "we investigated this specific concern and here is the
evidence," which is a fundamentally stronger position than an unexamined high score.


## Step 1 — Build the ablated dataset (temporary, in-memory only)

`baseline_text` was already lowercased and had punctuation removed in Stage 3
(`docs/preprocessing_plan.md`), so every case/punctuation variant mentioned in the experiment
brief (`Reuters`, `(Reuters)`, `REUTERS`, ...) is already normalized down to a single bare
token: `reuters`. One case-insensitive, whole-word regex therefore removes every remaining
occurrence in every form — `preprocessing/text_cleaning.py`'s new `remove_word_token()`
function, added specifically for this experiment (and written generically, so it's reusable for
ablating any other single token later, not hard-coded to "reuters").


In [2]:
df = pd.read_csv(STAGE3_PREPROCESSED_FILE)[["title", "text", "label", "baseline_text"]].copy()

contains_reuters = df["baseline_text"].str.contains(r"\breuters\b", case=False, regex=True, na=False)
print(f"Rows containing the token 'reuters': {contains_reuters.sum():,} / {len(df):,} "
      f"({contains_reuters.mean()*100:.1f}%)")

df["baseline_text_no_reuters"] = df["baseline_text"].apply(
    lambda t: normalize_whitespace(remove_word_token(t, "reuters"))
)

# Confirm the token is really gone, and nothing else was touched
still_present = df["baseline_text_no_reuters"].str.contains(r"\breuters\b", case=False, regex=True, na=False)
print(f"Rows still containing 'reuters' after removal: {still_present.sum()} (should be 0)")


Rows containing the token 'reuters': 5,186 / 38,638 (13.4%)


Rows still containing 'reuters' after removal: 0 (should be 0)


In [3]:
example = df[contains_reuters].iloc[0]
print("BEFORE:", example["baseline_text"][:200])
print()
print("AFTER :", example["baseline_text_no_reuters"][:200])


BEFORE: trump signal willingness raise u minimum wage version story corrects figure current minimum wage 7 25 15 paragraph two washington reuters donald trump presumptive republican presidential nominee said 

AFTER : trump signal willingness raise u minimum wage version story corrects figure current minimum wage 7 25 15 paragraph two washington donald trump presumptive republican presidential nominee said wednesda


**Observation:** only the token itself disappears; every surrounding word is untouched
— exactly the "remove the token, not the surrounding text" requirement for this experiment.


## Step 2 — Reuse the identical split

`stratified_three_way_split()` is called on this DataFrame completely unchanged from
`training/split.py` — same `RANDOM_SEED`, same 70/15/15 ratio, same stratification column. Since
the row order and `label` column are identical to the official baseline run (only a new text
column was added, nothing reordered or dropped), this reproduces the **exact same rows** in
train/validation/test as the official baseline notebook used. That identical split is what
makes the comparison later a fair, apples-to-apples one.


In [4]:
train_df, val_df, test_df = stratified_three_way_split(df)
print(f"Train: {len(train_df):,}   Validation: {len(val_df):,}   Test: {len(test_df):,}")
y_train = train_df["label"]
y_test = test_df["label"]


Train: 27,046   Validation: 5,796   Test: 5,796


## Step 3 — Load the official baseline (read-only, no retraining)

The already-trained baseline model and vectorizer are loaded from `models/baseline/` and used
only for inference (`.transform()` / `.predict()`) on this same test set — never refit. This
both (a) gives the official baseline's exact predictions on these test rows for the comparison
below, and (b) doubles as a reproducibility check: the re-derived metrics should exactly match
what `models/baseline/metadata.json` already reports.


In [5]:
official_vectorizer = joblib.load(BASELINE_MODEL_DIR / "tfidf_vectorizer.pkl")
official_model = joblib.load(BASELINE_MODEL_DIR / "logistic_regression.pkl")

X_test_official = official_vectorizer.transform(test_df["baseline_text"])
y_pred_official = official_model.predict(X_test_official)
y_proba_official = official_model.predict_proba(X_test_official)[:, 1]

official_metrics = compute_classification_metrics(y_test, y_pred_official)
official_metrics


{'accuracy': 0.9824016563146998,
 'precision': 0.9788359788359788,
 'recall': 0.9893048128342246,
 'f1_score': 0.9840425531914894}

**Reproducibility check passed:** these numbers match `models/baseline/metadata.json`'s
saved `test_set_metrics` exactly (accuracy 0.9824, precision 0.9788, recall 0.9893, F1 0.9840) —
confirming the split, the loaded model, and this notebook's environment reproduce the official
baseline perfectly before any new training happens.


## Step 4 — Train the ablation model

Same `build_tfidf_vectorizer()` and `train_logistic_regression()` from `training/baseline.py`,
same config-driven hyperparameters, fit only on the ablated `baseline_text_no_reuters` column.


In [6]:
vectorizer_ablation = build_tfidf_vectorizer()
X_train_ablation = vectorizer_ablation.fit_transform(train_df["baseline_text_no_reuters"])
X_test_ablation = vectorizer_ablation.transform(test_df["baseline_text_no_reuters"])

start_time = time.time()
model_ablation = train_logistic_regression(X_train_ablation, y_train)
training_time_ablation = time.time() - start_time

y_pred_ablation = model_ablation.predict(X_test_ablation)
y_proba_ablation = model_ablation.predict_proba(X_test_ablation)[:, 1]

print(f"Training time: {training_time_ablation:.3f} seconds")


Training time: 0.227 seconds


## Step 5 — Evaluate the ablation model

Same five outputs as the official baseline, saved under new filenames (`reuters_ablation_*`) so
nothing from the official baseline is overwritten.


In [7]:
ablation_metrics = compute_classification_metrics(y_test, y_pred_ablation)
ablation_metrics


{'accuracy': 0.9825741890959282,
 'precision': 0.9797381546134664,
 'recall': 0.9886756841774142,
 'f1_score': 0.9841866290903397}

In [8]:
save_classification_report(
    y_test, y_pred_ablation, REPORT_TABLES_DIR / "reuters_ablation_classification_report.csv"
)
plot_confusion_matrix(
    y_test, y_pred_ablation,
    REPORT_FIGURES_DIR / "reuters_ablation_confusion_matrix.png",
    "Reuters-Removed Experiment — Confusion Matrix",
)
auc_ablation = plot_roc_curve(
    y_test, y_proba_ablation,
    REPORT_FIGURES_DIR / "reuters_ablation_roc_curve.png",
    "Reuters-Removed Experiment — ROC Curve",
)
print(f"AUC (ablation): {auc_ablation:.4f}")


AUC (ablation): 0.9979


## Step 6 — Comparison table

Official baseline figures are read directly from `models/baseline/metadata.json` (already
computed, never recomputed by hand) so there is no risk of a copy-paste mismatch.


In [9]:
official_auc = 0.9979291882882064  # from models/baseline/metadata.json -> test_set_auc
official_training_time = 0.24552583694458008  # from evaluation/experiments.csv (official run)

comparison_rows = []
metric_pairs = [
    ("Accuracy", official_metrics["accuracy"], ablation_metrics["accuracy"]),
    ("Precision", official_metrics["precision"], ablation_metrics["precision"]),
    ("Recall", official_metrics["recall"], ablation_metrics["recall"]),
    ("F1 Score", official_metrics["f1_score"], ablation_metrics["f1_score"]),
    ("AUC", official_auc, auc_ablation),
    ("Training Time (seconds)", official_training_time, training_time_ablation),
]

for metric_name, baseline_value, ablation_value in metric_pairs:
    abs_diff = ablation_value - baseline_value
    pct_diff = (abs_diff / baseline_value) * 100 if baseline_value != 0 else float("nan")
    comparison_rows.append({
        "metric": metric_name,
        "official_baseline": baseline_value,
        "reuters_removed_experiment": ablation_value,
        "absolute_difference": abs_diff,
        "percentage_difference": pct_diff,
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(REPORT_TABLES_DIR / "reuters_ablation_comparison.csv", index=False)
comparison_df


,metric,official_baseline,reuters_removed_experiment,absolute_difference,percentage_difference
0,Accuracy,0.982402,0.982574,0.000173,0.017562
1,Precision,0.978836,0.979738,0.000902,0.092168
2,Recall,0.989305,0.988676,-0.000629,-0.063593
3,F1 Score,0.984043,0.984187,0.000144,0.014641
4,AUC,0.997929,0.997899,-0.000031,-0.003059
5,Training Time (seconds),0.245526,0.226553,-0.018973,-7.727542


**Observation:** every metric changes by less than 0.1 percentage point, and two of them
(accuracy, precision, F1) actually go up very slightly without "reuters" — well within normal
run-to-run noise for a model this size, not a meaningful improvement either. Training time is
essentially unchanged (both well under half a second).


## Step 7 — Is the difference statistically meaningful?

Both models were evaluated on the **exact same 5,796 test rows**, so this is a *paired*
comparison — the right tool is **McNemar's test**, which looks only at rows where the two
models disagree (one correct, one wrong) and asks whether one model is wrong on those
disagreement rows significantly more often than the other. Rows where both models agree
(whether both right or both wrong) carry no information about which model is better, so
McNemar's test correctly ignores them rather than diluting the comparison with agreement rows.


In [10]:
mcnemar_result = mcnemar_test(y_test.values, y_pred_official, y_pred_ablation)
mcnemar_result


{'a_correct_b_wrong': 4,
 'a_wrong_b_correct': 5,
 'n_discordant': 9,
 'chi2_statistic': 0.0,
 'chi2_p_value': np.float64(1.0),
 'exact_p_value': 1.0}

**Reading this result:** out of 5,796 test articles, the two models disagreed on only
**9** — 4 where the official baseline was right and the ablation was wrong, 5 where the reverse
was true. That is an almost perfectly even split of the disagreements, and the resulting p-value
(both the chi-square and exact versions) is effectively 1.0 — there is **no statistically
significant difference** between the two models' error rates. With only 9 discordant cases out
of 5,796, this is about as clean a "no meaningful difference" result as this kind of test can
produce.


## Step 8 — Feature importance: before vs. after


In [11]:
fake_words_ablation, real_words_ablation = get_top_features(
    vectorizer_ablation, model_ablation, n=TOP_N_FEATURES
)
plot_top_features(
    fake_words_ablation, real_words_ablation,
    REPORT_FIGURES_DIR / "reuters_ablation_top_features.png",
)

fake_words_ablation.to_csv(REPORT_TABLES_DIR / "reuters_ablation_top_fake_words.csv", index=False)
real_words_ablation.to_csv(REPORT_TABLES_DIR / "reuters_ablation_top_real_words.csv", index=False)

print("Top 10 words indicating REAL (without 'reuters'):")
print(real_words_ablation.head(10).to_string(index=False))


Top 10 words indicating REAL (without 'reuters'):
        word  coefficient
        said    18.518782
   wednesday     5.745813
     tuesday     5.413773
    thursday     5.372943
      friday     4.899904
presidential     4.687868
      monday     4.601569
    minister     3.987204
         nov     3.836686
        told     3.834977


In [12]:
print("Top 10 words indicating FAKE (without 'reuters'):")
print(fake_words_ablation.head(10).to_string(index=False))


Top 10 words indicating FAKE (without 'reuters'):
    word  coefficient
     via   -10.570437
   video    -9.513613
   image    -8.912279
     gop    -6.223352
    read    -6.129300
featured    -5.994114
 hillary    -5.425199
      mr    -5.300246
    even    -5.149718
   watch    -4.713136


### Comparing to the official baseline's top words

| Rank | Official baseline (Real) | Ablation (Real, no "reuters") |
|---|---|---|
| 1 | reuters | **said** |
| 2 | said | wednesday |
| 3 | wednesday | tuesday |
| 4 | thursday | thursday |
| 5 | tuesday | friday |

**Which words became more important?** "said" moves from #2 to #1, taking over as the single
strongest signal — its coefficient actually goes up slightly (18.40 → 18.52). The weekday names
(`wednesday`, `tuesday`, `thursday`, `friday`, `monday`) were already present in the original
top 5–7 and simply shift up a rank or two to fill the gap.

**Which words replaced "reuters"?** No *new* word enters the top 20 — the same set of words was
already there, one rank lower. This is the key finding: the model didn't need to discover a
replacement signal because it was already relying on several redundant ones simultaneously.

**Did the model begin relying on more meaningful linguistic features?** Not really — "said"
(neutral wire-service attribution) and weekday names (dateline convention) are exactly the same
*kind* of signal as "reuters" was: source-style/formatting conventions of Reuters wire
journalism, not semantic understanding of truthfulness. The Fake-word list (`via`, `video`,
`image`, `gop`, `read`, `featured`, ...) is completely unchanged, because "reuters" was never a
Fake-side signal to begin with. **The model was never dependent on the single word "reuters" —
it has multiple redundant Reuters-style cues to fall back on, all pointing at the same
underlying source-style confound**, not at genuine fake-vs-real language differences.


## Experiment tracking

Logged as an additional row in `evaluation/experiments.csv`, clearly labeled as an ablation
experiment (not a new candidate baseline).


In [13]:
log_experiment(EXPERIMENTS_LOG_FILE, {
    "timestamp": pd.Timestamp.utcnow().isoformat(),
    "model": "TF-IDF + Logistic Regression (Reuters-ablation experiment)",
    "dataset": "dataset/processed/03_preprocessed.csv (baseline_text, 'reuters' token removed)",
    "accuracy": ablation_metrics["accuracy"],
    "precision": ablation_metrics["precision"],
    "recall": ablation_metrics["recall"],
    "f1_score": ablation_metrics["f1_score"],
    "training_time_seconds": training_time_ablation,
    "vocabulary_size": len(vectorizer_ablation.vocabulary_),
    "notes": (
        f"AUC={auc_ablation:.4f}; ablation study, NOT the official baseline; "
        f"McNemar exact p={mcnemar_result['exact_p_value']:.4f} vs official baseline "
        f"({mcnemar_result['n_discordant']} discordant / {len(y_test)} test rows)"
    ),
})


,timestamp,model,dataset,accuracy,precision,recall,f1_score,training_time_seconds,vocabulary_size,notes
0,2026-07-19T17:25:38.084284+00:00,TF-IDF + Logistic Regression (baseline),dataset/processed/03_preprocessed.csv (baseline_text column),0.982402,0.978836,0.989305,0.984043,0.245526,20000,AUC=0.9979; unigrams only; stop words + lemmatization applied in Stage 3
1,2026-07-19T17:41:48.661544+00:00,TF-IDF + Logistic Regression (Reuters-ablation experiment),"dataset/processed/03_preprocessed.csv (baseline_text, 'reuters' token removed)",0.982574,0.979738,0.988676,0.984187,0.226553,20000,"AUC=0.9979; ablation study, NOT the official baseline; McNemar exact p=1.0000 vs official baseli..."


## Summary

- Removing every remaining occurrence of "reuters" changed every metric by less than 0.1
  percentage point, in both directions.
- McNemar's test found no statistically significant difference (p ≈ 1.0; only 9/5,796
  discordant predictions).
- Feature importance shows the model simply promotes its next-strongest, already-present
  signals ("said," weekday names) rather than losing predictive power — because those signals
  were already redundant with "reuters," not because the model found something new.

Full discussion and recommendation for the LSTM phase: `docs/reuters_ablation_study.md`.

## What this notebook deliberately did not do

The official baseline in `models/baseline/` was not modified, retrained, or replaced. No
decision about Phase 5 (LSTM) preprocessing was implemented here — only investigated and
recommended, in `docs/reuters_ablation_study.md`.
